# 시각 의미 데이터 분석 재현

2026-10-02 소스 스냅샷. 저장소 루트를 작업 디렉터리로 사용한다. 원본 해시와 현재 레지스트리를 직접 확인하고, 저장된 resolver 진단을 집계한다. 소스에서 파생한 진단은 독립 holdout이나 이미지 품질 측정이 아니다.


In [1]:
import collections, csv, hashlib, json, sys
from pathlib import Path
ROOT = Path.cwd()
OUT = ROOT / "docs/analysis/2026-10-02-visual-semantics-data-audit"
metrics = json.loads((OUT / "metrics.json").read_text())
manifest = json.loads((OUT / "source-manifest.json").read_text())
inventory = list(csv.DictReader((OUT / "profile-inventory.csv").open()))
print("commit:", metrics["commit"])
print("captured_at_kst:", metrics["captured_at_kst"])


commit: 0e5cc0d1968d5cfe83d1b63e9cfa198724ea9463
captured_at_kst: 2026-10-02T11:11:37.198491+09:00


In [2]:
source_counts = []
for entry in manifest["files"]:
    path = ROOT / entry["path"]
    if path.name.startswith("photo_prompt_visual_obligations") and path.suffix == ".json":
        source_counts.append((path.name, len(json.loads(path.read_text())["profiles"])))
profiles = sum(count for _, count in source_counts)
groups = sum(int(row["component_groups"]) for row in inventory)
assert len(source_counts) == metrics["totals"]["files"]
assert profiles == len(inventory) == metrics["totals"]["compiled_profiles"]
assert groups == metrics["totals"]["component_groups"]
print(json.dumps({"registry_files": len(source_counts), "profiles": profiles,
                  "component_groups": groups}, ensure_ascii=False))


{"registry_files": 23, "profiles": 1385, "component_groups": 3289}


In [3]:
mismatches = [entry["path"] for entry in manifest["files"]
              if hashlib.sha256((ROOT / entry["path"]).read_bytes()).hexdigest() != entry["sha256"]]
assert not mismatches, mismatches
print("Source hashes unchanged:", len(manifest["files"]))
print("Existing index validation:", metrics["structure_checks"])


Source hashes unchanged: 30
Existing index validation: {'registry_index_metadata_check': 'pass', 'registry_hash_matches': True, 'duplicate_profile_ids': 0, 'cross_profile_gate_id_collisions': 0, 'exact_term_collision_groups': 0}


In [4]:
distribution = metrics["source_distribution"]
groups_total = metrics["totals"]["component_groups"]
chart_rows = []
labels = ["기본", "Y2K", "의복", "코스튬", "장신구", "수영복"]
grouped = [(label, row["profiles"], row["component_groups"])
           for label, row in zip(labels, distribution[:6])]
grouped.append(("기타", sum(row["profiles"] for row in distribution[6:]),
                sum(row["component_groups"] for row in distribution[6:])))
for family, profile_count, component_count in grouped:
    for basis, count, denominator in [("프로필 비중", profile_count, profiles),
                                      ("구성 요소 비중", component_count, groups_total)]:
        chart_rows.append({"family": family, "basis": basis, "count": count,
                           "denominator": denominator,
                           "share": round(count / denominator * 100, 2)})
assert sum(count for _, count, _ in grouped) == profiles
assert sum(count for _, _, count in grouped) == groups_total
print(json.dumps(chart_rows, ensure_ascii=False, indent=2))


[
  {
    "family": "기본",
    "basis": "프로필 비중",
    "count": 333,
    "denominator": 1385,
    "share": 24.04
  },
  {
    "family": "기본",
    "basis": "구성 요소 비중",
    "count": 1539,
    "denominator": 3289,
    "share": 46.79
  },
  {
    "family": "Y2K",
    "basis": "프로필 비중",
    "count": 306,
    "denominator": 1385,
    "share": 22.09
  },
  {
    "family": "Y2K",
    "basis": "구성 요소 비중",
    "count": 313,
    "denominator": 3289,
    "share": 9.52
  },
  {
    "family": "의복",
    "basis": "프로필 비중",
    "count": 157,
    "denominator": 1385,
    "share": 11.34
  },
  {
    "family": "의복",
    "basis": "구성 요소 비중",
    "count": 160,
    "denominator": 3289,
    "share": 4.86
  },
  {
    "family": "코스튬",
    "basis": "프로필 비중",
    "count": 87,
    "denominator": 1385,
    "share": 6.28
  },
  {
    "family": "코스튬",
    "basis": "구성 요소 비중",
    "count": 87,
    "denominator": 3289,
    "share": 2.65
  },
  {
    "family": "장신구",
    "basis": "프로필 비중",
    "count": 68,
    "denominat

In [5]:
flags = metrics["profile_flags"]
guard = metrics["discovery_guard_details"]
language_rows = [
    {"metric": "한국어 exact/alias 존재", "count": flags["exact_has_korean"], "denominator": profiles},
    {"metric": "한국어 positive 검색 텍스트 존재", "count": flags["semantic_text_has_korean"], "denominator": profiles},
    {"metric": "모든 구성 요소 그룹에 한국어 대안 존재", "count": flags["all_component_groups_have_korean"], "denominator": profiles},
    {"metric": "guarded 중 한국어 구성 요소 대안 전무", "count": guard["guarded_with_no_korean_component_terms"], "denominator": guard["guarded_profiles"]},
]
for row in language_rows:
    row["share_pct"] = round(row["count"] / row["denominator"] * 100, 1)
print(json.dumps(language_rows, ensure_ascii=False, indent=2))


[
  {
    "metric": "한국어 exact/alias 존재",
    "count": 850,
    "denominator": 1385,
    "share_pct": 61.4
  },
  {
    "metric": "한국어 positive 검색 텍스트 존재",
    "count": 1319,
    "denominator": 1385,
    "share_pct": 95.2
  },
  {
    "metric": "모든 구성 요소 그룹에 한국어 대안 존재",
    "count": 620,
    "denominator": 1385,
    "share_pct": 44.8
  },
  {
    "metric": "guarded 중 한국어 구성 요소 대안 전무",
    "count": 272,
    "denominator": 833,
    "share_pct": 32.7
  }
]


## 자체 구성 요소와 제외 조건의 일관성

현재 운영 함수를 실행한다. 그룹 첫 표현을 이어 만든 문맥은 소스 기반 진단이며 실제 스킬의 공개 authorial core를 대신하지 않는다.


In [6]:
sys.path.insert(0, str(ROOT / "skills/photo-prompt-image-generator/scripts"))
import prompt_generator as pg
registry = pg.load_visual_obligation_registry(ROOT / "skills/photo-prompt-image-generator/assets/photo_prompt_visual_obligations.json")
conflict_rows = []
for profile in registry["profiles"]:
    context = "; ".join(group["any_terms"][0] for group in profile["semantics"]["component_semantics"]["groups"])
    eligible, reason = pg.visual_profile_context_applicability(
        profile, context, has_authorial_core_context=True, require_positive_context_terms=False)
    if not eligible and reason == "request_exclusion":
        conflict_rows.append({"profile_id": profile["id"], "reason": reason})
recorded = json.loads((OUT / "source-component-context-conflicts.json").read_text())
assert {row["profile_id"] for row in conflict_rows} == {row["profile_id"] for row in recorded}
print("Self-context request exclusions:", len(conflict_rows), "/", len(registry["profiles"]))
print(json.dumps(conflict_rows, ensure_ascii=False, indent=2))


Self-context request exclusions: 21 / 1385
[
  {
    "profile_id": "wide_angle_near_field_perspective",
    "reason": "request_exclusion"
  },
  {
    "profile_id": "panning_subject_tracking_motion_relation",
    "reason": "request_exclusion"
  },
  {
    "profile_id": "rear_curtain_flash_motion_trace",
    "reason": "request_exclusion"
  },
  {
    "profile_id": "negative_fill_shadow_deepening_relation",
    "reason": "request_exclusion"
  },
  {
    "profile_id": "diffusion_filter_highlight_halation",
    "reason": "request_exclusion"
  },
  {
    "profile_id": "mixed_illuminant_white_balance_relation",
    "reason": "request_exclusion"
  },
  {
    "profile_id": "highlight_rolloff_tone_response",
    "reason": "request_exclusion"
  },
  {
    "profile_id": "loop_face_light_pattern",
    "reason": "request_exclusion"
  },
  {
    "profile_id": "split_face_light_pattern",
    "reason": "request_exclusion"
  },
  {
    "profile_id": "low_key_selective_illumination",
    "reason": "requ

## Resolver 진단 집계

조회 결과는 retrieval_probes.py가 실제 resolver를 실행해 저장한 28건이다. 여기서는 전체 resolver를 다시 돌리지 않고 결과를 집계한다. 질의 임베딩 API, 후보 채택, 생성·픽셀 평가는 포함하지 않는다.


In [7]:
probes = json.loads((OUT / "retrieval-probes.json").read_text())
assert probes["completed"] is True
assert len(probes["cases"]) == 28
probe_summary = []
for tier in ["natural_request_only", "source_derived_exact_control", "source_derived_component_control", "broad_or_negative_control"]:
    cases = [case for case in probes["cases"] if case["tier"] == tier]
    probe_summary.append({"tier": tier, "cases": len(cases),
        "target_hits": sum(case["target_hit"] is not None for case in cases),
        "target_hard": sum(bool(case["target_hit"] and case["target_hit"]["hard_eligible"]) for case in cases),
        "any_hard": sum(bool(case["hard_profile_ids"]) for case in cases)})
print(json.dumps(probe_summary, ensure_ascii=False, indent=2))


[
  {
    "tier": "natural_request_only",
    "cases": 8,
    "target_hits": 5,
    "target_hard": 1,
    "any_hard": 1
  },
  {
    "tier": "source_derived_exact_control",
    "cases": 8,
    "target_hits": 8,
    "target_hard": 8,
    "any_hard": 8
  },
  {
    "tier": "source_derived_component_control",
    "cases": 8,
    "target_hits": 7,
    "target_hard": 1,
    "any_hard": 1
  },
  {
    "tier": "broad_or_negative_control",
    "cases": 4,
    "target_hits": 0,
    "target_hard": 0,
    "any_hard": 0
  }
]


In [8]:
pairs = json.loads((OUT / "near-duplicate-vector-pairs.json").read_text())
contrast_pairs = [{"left": row["left"], "right": row["right"], "cosine": row["cosine"]}
                  for row in pairs if row["cosine"] >= 0.95]
assert len(contrast_pairs) == metrics["vector_checks"]["pair_counts_by_threshold"]["0.95"] == 6
print(json.dumps(contrast_pairs, ensure_ascii=False, indent=2))


[
  {
    "left": "cr_warm_highlights",
    "right": "cr_cool_highlights",
    "cosine": 0.98793618
  },
  {
    "left": "top_hourglass_silhouette_relation",
    "right": "bottom_hourglass_silhouette_relation",
    "cosine": 0.97532009
  },
  {
    "left": "cr_warm_foreground",
    "right": "cr_cool_foreground",
    "cosine": 0.97493519
  },
  {
    "left": "broad_face_light_orientation_relation",
    "right": "short_face_light_orientation_relation",
    "cosine": 0.97054666
  },
  {
    "left": "cr_colored_key",
    "right": "cr_colored_fill",
    "cosine": 0.96647834
  },
  {
    "left": "cr_complex_subject",
    "right": "cr_complex_background",
    "cosine": 0.9639931
  }
]


## 해석과 개선 순서

자기 모순 21개를 먼저 고치고, 한국어 구성 요소 표현과 반대 의미 대조를 보강한다. 높은 벡터 유사도는 실제 오검색률이나 중복의 증거가 아니다. 수요 자료 없이 분야별 개수를 같은 비율로 늘리지 않는다. 다음 검증에서는 요청과 baseline을 데이터 조회 전에 고정하고, 등록·검색·채택·프롬프트·픽셀·사용자 수용을 구별한다.
